# Aula 05 — como treinar uma cabeça sobre um modelo congelado

**Objetivo:** compreender o que é treinado, como os dados orientam a atualização dos pesos e como verificar se o resultado é útil. Ao final, você deverá conseguir explicar o percurso de uma entrada até sua classificação e distinguir treinamento, calibração e inferência.

Usaremos o Phi-4 como produtor de representações e treinaremos duas pequenas camadas de saída, chamadas **cabeças**. Os dados de um projeto existente tornam o processo executável; não é necessário aprender as regras desse projeto para compreender o método.

**Percurso principal:** execute as seções 1 a 10 em sequência para realizar o treinamento. A extração de representações dos 816 exemplos é a etapa demorada. **Demonstração sem esperar o treinamento:** a seção 11 tem uma única célula independente que usa a cabeça já treinada do GitHub. Você pode executar essa célula em uma sessão nova, sem executar as anteriores.


## 1. O que significa “treinar uma cabeça”?

Um modelo pré-treinado transforma texto em uma representação numérica. Nesta aula, mantemos esse modelo **congelado**: seus pesos não são atualizados. Treinamos uma camada que aprende a transformar a representação em uma previsão adequada ao nosso alvo.

```text
Texto → tokenização → Phi-4 CONGELADO → vetor com 5.120 valores
                                                  │
                         ┌────────────────────────┴────────────────────┐
                         ↓                                             ↓
               Cabeça de classificação                      Cabeça probabilística
                 3 escores → softmax                          1 escore → sigmoid
                         ↓                                             ↓
                   classe prevista                             p(A) e 1 − p(A)
```

| Conceito | Papel no processo | Neste experimento |
|---|---|---|
| Entrada `x` | Informação disponível para produzir a previsão. | Texto estruturado por um contrato. |
| Alvo `y` | Resposta usada como supervisão no treinamento. | Classe discreta ou frequência de um evento. |
| Representação `h` | Saída numérica do modelo base. | Último estado oculto do prompt completo, com 5.120 valores. |
| Cabeça `W, b` | Parâmetros que aprendemos. | Uma camada linear independente por tarefa. |
| Perda | Mede o desacordo entre previsão e alvo. | Entropia cruzada ou perda binária. |
| Otimizador | Atualiza os parâmetros a partir dos gradientes. | Adam. |

**Por que treinar apenas a cabeça?** Neste método, reutilizamos as representações do Phi-4 e reduzimos a quantidade de parâmetros atualizados. Isso torna o treino da camada final pequeno; não torna barata a execução do Phi-4 nem garante generalização. Não é fine-tuning de todos os pesos ou LoRA.

A cabeça de classificação tem `3 × 5.120 + 3 = 15.363` parâmetros; a probabilística tem `1 × 5.120 + 1 = 5.121`. São dois problemas de aprendizagem separados, com alvos e perdas diferentes.

**Antes de continuar:** qual componente precisa mudar quando a previsão está errada? Nesta aula, a atualização alcança `W` e `b`, não os pesos do Phi-4.


## 2. Preparar o ambiente

**O que faremos:** instalar as bibliotecas e definir onde serão salvos dados, cache de representações e cabeças. **Por quê:** a extração é demorada e deve poder ser retomada.

No Colab, usamos o PyTorch e o NumPy disponíveis. Instalamos apenas as versões específicas usadas pelo carregador. O corpo do modelo usa o dispositivo selecionado; o treino das cabeças ocorre na CPU. Para persistir entre sessões do Colab, monte seu Drive e altere `WORK_DIR` para uma pasta nele antes de iniciar. O corpo Phi-4 é baixado na revisão fixa do contrato. A janela é de 16.384 tokens e não há truncamento.

**Observe após executar:** a pasta de trabalho exibida. Ela deve ter espaço para os pesos base e os artefatos produzidos.


In [ ]:
!pip install -q transformers==5.0.0 safetensors==0.8.0 huggingface-hub==1.23.0


In [ ]:
from pathlib import Path
WORK_DIR = Path("treinamento-aula05")
WORK_DIR.mkdir(parents=True, exist_ok=True)


## 3. Obter entradas, alvos e código

**O que faremos:** clonar a `main` do GitHub e abrir o pacote de dados e scripts. **Por quê:** precisamos reproduzir o mesmo contrato e preservar o alinhamento entre cada entrada e seu alvo.

A pasta `treinamento/piloto-v01/pacote` contém scripts e dados. O checkout usa um único commit da `main`; os carregadores conferem os hashes e o contrato do pacote. O corpo Phi-4 é baixado separadamente.

| Arquivo | Conteúdo | Entra no prompt? |
|---|---|---|
| `dados/inputs.jsonl.gz` | Entradas das duas tarefas. | Sim, através do encoder de cada tarefa. |
| `dados/supervisao.jsonl` | Alvo, tarefa, partição e grupo de cada exemplo. | Não. |
| `protocolo.json` | Partições, hiperparâmetros e critérios de calibração. | Não. |
| `modelo.json` | Revisão, arquivos e contrato da representação. | Não; configura o carregamento. |

**Observe após executar:** são 816 exemplos alinhados. O mesmo índice liga entrada e supervisão para treinamento; ele não é uma informação fornecida ao modelo.


In [ ]:
!git clone -q --depth 1 --branch main https://github.com/AIConnection/missao-espacial.git missao-espacial

import json, gzip, hashlib
from pathlib import Path
import numpy as np
REPO_DIR = Path("missao-espacial")
PACKAGE_DIR = REPO_DIR / "treinamento/piloto-v01/pacote"
# Conferir os arquivos, antes de separar entrada e supervisão.
integridade = json.loads((PACKAGE_DIR / "integridade.json").read_text())
for nome, esperado in integridade.items():
    assert hashlib.sha256((PACKAGE_DIR / nome).read_bytes()).hexdigest() == esperado, nome
with gzip.open(PACKAGE_DIR / "dados/inputs.jsonl.gz", "rt", encoding="utf-8") as arquivo:
    inputs = [json.loads(linha) for linha in arquivo]
supervision = [json.loads(linha) for linha in (PACKAGE_DIR / "dados/supervisao.jsonl").read_text().splitlines()]
CLASSES = ["suporte", "insuficiente", "contradição"]
LABELS = CLASSES
assert len(inputs) == len(supervision) == 816
assert [r["index"] for r in supervision] == list(range(816))
print("Entradas:", len(inputs), "| Alvos separados:", len(supervision))


### 3.1. Separar entrada e supervisão

Vamos inspecionar os **cinco primeiros e os cinco últimos registros da partição de treino de cada tarefa**. Cada entrada aparece completa, seguida do alvo e dos metadados em outro bloco. Não mostramos as 816 entradas; não alteramos nem truncamos as entradas usadas pelo modelo.

Nesta tarefa, a entrada contém `premise` (evidência disponível) e `hypothesis` (afirmação avaliada). O alvo é uma relação: suporte, insuficiente ou contradição. Essa semântica define **a tarefa de aprendizagem**, sem exigir que a aula ensine as regras do ambiente usado para gerar os textos.

O carregador acrescenta um prefixo constante à premissa e usa a instrução e o chat template originais. Eles fazem parte do contrato da representação e permanecem iguais no treino e na inferência. A instrução original menciona A/B/C, mas a cabeça treinada não gera letras: ela classifica a representação.

**Observe:** entrada e alvo são mostrados em blocos diferentes. O alvo e os metadados nunca são concatenados ao texto. Se entregássemos o gabarito na entrada, uma boa métrica poderia refletir vazamento em vez de aprendizagem.

Na supervisão, `target=0` significa suporte, `target=1` significa insuficiente e `target=2` significa contradição. Essa é também a ordem das três saídas da cabeça. As chaves `task`, `split`, `group` e `index` organizam o experimento e ficam fora da entrada.


In [ ]:
def mostrar_amostra(entradas, alvos, indices, titulo):
    print("\n" + titulo)
    selecionados = list(dict.fromkeys(indices[:5] + indices[-5:]))
    for indice in selecionados:
        print(f"\nREGISTRO {indice} — ENTRADA COMPLETA:")
        print(json.dumps(entradas[indice], ensure_ascii=False, indent=2))
        print("ALVO E METADADOS — FORA DO INPUT:")
        print(json.dumps(alvos[indice], ensure_ascii=False, indent=2))

for tarefa in ("nli", "forecast"):
    indices = [r["index"] for r in supervision if r["task"] == tarefa and r["split"] == "train"]
    mostrar_amostra(inputs, supervision, indices, tarefa + " — cinco primeiros e cinco últimos do TREINO")


### 3.2. Classes discretas e alvos probabilísticos

| Aspecto | Classificação | Estimativa de evento |
|---|---|---|
| Alvo de um exemplo | Uma classe entre três. | Frequência `k / n` entre 0 e 1. |
| Saída interna | Três probabilidades cuja soma é 1. | `p(A)`; a probabilidade de B é `1 − p(A)`. |
| Perda de treino | Entropia cruzada multiclasse. | Entropia cruzada binária com alvo fracionário. |
| Saída pública | Apenas a classe escolhida. | As duas probabilidades dos eventos contratados. |

Um alvo `28/32 = 0,875` significa que o evento A ocorreu em 28 das 32 continuações simuladas sob condições definidas. Não é uma certeza nem uma probabilidade universal. A cabeça aprende a aproximar essa frequência condicionada à entrada e ao procedimento de amostragem.

A identidade de A e B pertence ao contrato do projeto: A é entrega e B é retorno com sobrevivência. Para aprender o processo, o ponto importante é que são os dois resultados exclusivos representados e que **o alvo vem de contagens observadas**, não de uma opinião do modelo. A entrada desta tarefa contém contexto, objetivo, plano candidato, evento-alvo e condições de amostragem; estes textos condicionam a previsão, mas não constituem uma política de ações.

**Observe na célula seguinte:** distribuição de classes, distribuição das frequências e os dados de contagem de um exemplo. Uma massa de frequências perto de 1 pode tornar uma constante difícil de superar. Balancear classes na primeira tarefa não balanceia os eventos da segunda.


In [ ]:
from collections import Counter
print("CLASSIFICAÇÃO — quantidade por classe:")
print(dict(Counter(LABELS[r["target"]] for r in supervision if r["task"] == "nli")))
print("\nPROBABILIDADES — quantidade de exemplos por alvo k/n:")
print(dict(sorted(Counter(r["target"] for r in supervision if r["task"] == "forecast").items())))
exemplo_prob = next(r for r in supervision if r["task"] == "forecast" and 0 < r["target"] < 1)
print("\nUM ALVO REAL, SEPARADO DA ENTRADA:")
print(json.dumps({"target": exemplo_prob["target"], "counts": exemplo_prob["supervision"]["counts"],
                  "trials": exemplo_prob["supervision"]["trials"]}, ensure_ascii=False, indent=2))


## 4. Separar treino, calibração e teste

```text
Treino          → calcula normalização e atualiza W e b
Calibração      → escolhe temperatura com W e b já fixos
Teste reservado → mede resultado; não escolhe parâmetros
```

| Tarefa | Treino | Calibração | Teste | Total |
|---|---:|---:|---:|---:|
| Classificação | 420 | 90 | 90 | 600 |
| Frequência de evento | 144 | 36 | 36 | 216 |

Na classificação, cada partição é balanceada: 140 por classe no treino e 30 por classe em cada reserva. Exemplos relacionados são mantidos no mesmo grupo e um grupo não atravessa partições. Para a tarefa probabilística, as variantes de cada família estrutural ficam juntas.

**Por que separar por grupo?** Se uma versão de um exemplo entrar no treino e uma versão quase igual no teste, podemos medir reconhecimento de um padrão repetido. A separação por grupo reduz esse vazamento; não garante que o teste cubra todos os assuntos ou estratégias. As 6.912 continuações simuladas sustentam 216 alvos e não são 6.912 exemplos independentes de treino.

**O que faremos:** conferir os tamanhos das partições e os hiperparâmetros definidos antes da avaliação.


In [ ]:
protocol = json.loads((PACKAGE_DIR / "protocolo.json").read_text())
print(dict(Counter((r["task"], r["split"]) for r in supervision)))
for tarefa in ("nli", "forecast"):
    grupos = {particao: {r["group"] for r in supervision if r["task"] == tarefa and r["split"] == particao}
              for particao in ("train", "calibration", "test")}
    assert not (grupos["train"] & grupos["calibration"] or grupos["train"] & grupos["test"] or grupos["calibration"] & grupos["test"])
print("Grupos separados; hiperparâmetros:", protocol["optimization"])


## 5. Extrair representações do Phi-4 congelado

**O que faremos:** primeiro baixar os arquivos do Phi-4; depois carregá-lo e executá-lo para cada entrada e salvar o último estado oculto do prompt completo. Cada entrada vira um vetor de 5.120 números; nenhuma supervisão é acrescentada ao prompt.

```text
Entrada + protocolo fixo → tokens → Phi-4 congelado → h ∈ R⁵¹²⁰ → cache
```

**Por que salvar as representações?** O corpo não muda entre épocas; podemos executar a etapa cara uma vez e reutilizar os vetores ao treinar as cabeças. O cache não é a cabeça e seus números não são probabilidades.

A extração também captura a projeção original A/B/C, usada como baseline da classificação. Essa projeção não é uma cabeça treinada em outro projeto.

**Observe:** ao concluir, o cache deve ter 816 linhas. A retomada verifica identidade do modelo, ambiente e hashes; não usa um cache parcial para treinar. Mudar dispositivo, runtime ou configuração exige outra pasta de trabalho. Essa etapa pode levar horas; para demonstrar o resultado sem esperar, vá à seção 11.

### 5.1. Baixar o corpo Phi-4

A clonagem do GitHub trouxe as cabeças treinadas, os dados e os scripts. **Os pesos base do Phi-4 vêm do Hugging Face**, do repositório e revisão registrados em `modelo.json`. A célula abaixo faz esse download explicitamente. São cerca de 8,2 GB; arquivos já em cache são reutilizados.

**Observe:** nome do modelo, revisão, progresso do download e pasta final. Ter arquivos no disco ainda não significa que o modelo foi carregado na memória ou executado.


In [ ]:
from huggingface_hub import snapshot_download
modelo_base = json.loads((PACKAGE_DIR / "modelo.json").read_text())
print("Phi-4 base:", modelo_base["repo"], "| Revisão:", modelo_base["revision"])
PHI4_DIR = snapshot_download(
    repo_id=modelo_base["repo"], revision=modelo_base["revision"],
    allow_patterns=[arquivo["path"] for arquivo in modelo_base["model_files"]])
print("Arquivos do Phi-4 no disco:", PHI4_DIR)


### 5.2. Carregar e executar o Phi-4 congelado

A seguir copiamos a implementação do carregador e instanciamos `Phi4Features` com a pasta baixada. Depois executamos explicitamente `corpo.features(tokens)` para cada entrada. Cada execução produz a representação que será salva no cache. Os pesos do Phi-4 ficam congelados.

```text
Arquivos do Phi-4 no disco → modelo carregado na memória → execução sobre tokens → vetor
```

O treinamento posterior das cabeças usa esses vetores: ele não precisa executar o Phi-4 novamente a cada época. A inferência de uma entrada nova precisa executar o corpo outra vez.


### Código do carregamento, visível nesta aula

A classe abaixo é uma cópia do carregador utilizado no projeto. Ela verifica os arquivos baixados, monta a arquitetura do Phi-4, lê os tensores quantizados e congela o corpo. O formato de 4 bits desses pesos requer este carregador; não trocamos o formato para simplificar a aula.

Observe em `Phi4Features.__init__`: criação de `Phi3Model`, leitura dos pesos, `eval()` e `requires_grad_(False)`. A célula seguinte instancia o corpo explicitamente. Os detalhes da desquantização estão disponíveis aqui, mas a aprendizagem da cabeça começa depois de o vetor ser produzido.

`Phi3Model` é o nome da classe de arquitetura usada neste carregador do Transformers. Os pesos e a configuração carregados são os do **Phi-4**, identificados em `modelo.json`; não estamos usando os pesos de um Phi-3.


In [ ]:
"""Mesmo Phi-4 quantizado, executado por PyTorch em CUDA, MPS ou CPU.
Não usa geração, bitsandbytes, FlashAttention obrigatório, MLX ou kernel Metal próprio.
"""
from pathlib import Path
import hashlib,json,platform
import numpy as np
import transformers
import torch
from torch import nn
from torch.nn import functional as F
from safetensors import safe_open
from transformers import AutoTokenizer,Phi3Config
from transformers.models.phi3.modeling_phi3 import Phi3Model,Phi3RotaryEmbedding

NLI_SYSTEM=('Classify the relation of a hypothesis to a premise using only the premise. '
 'A = entailment: the hypothesis follows from the premise. '
 'B = neutral: the hypothesis neither follows from nor contradicts the premise. '
 'C = contradiction: the hypothesis contradicts the premise. '
 'Answer with exactly one letter: A, B, or C.')

def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def automatic_device():
 if torch.cuda.is_available():return 'cuda'
 if hasattr(torch.backends,'mps') and torch.backends.mps.is_available():return 'mps'
 return 'cpu'
def choose_device(requested='auto'):
 value=automatic_device() if requested=='auto' else requested
 if value not in ('cpu','cuda','mps'):raise ValueError('Dispositivo desconhecido')
 if value=='cuda' and not torch.cuda.is_available():raise ValueError('CUDA indisponível; use auto ou cpu')
 if value=='mps' and not torch.backends.mps.is_available():raise ValueError('MPS indisponível; use auto ou cpu')
 return value

def prepare_assets(manifest_path,model_directory=None):
 manifest=json.loads(Path(manifest_path).read_text())
 if model_directory is None:
  from huggingface_hub import snapshot_download
  model_directory=snapshot_download(repo_id=manifest['repo'],revision=manifest['revision'],allow_patterns=[r['path'] for r in manifest['model_files']])
 directory=Path(model_directory)
 for r in manifest['model_files']:
  p=directory/r['path']
  if not p.is_file() or p.stat().st_size!=r['bytes'] or sha(p)!=r['sha256']:raise ValueError(f'Peso/tokenizer ausente ou incompatível: {p}')
 return directory,manifest

def unpack(weight,scales,biases,dtype):
 """Grupos de 64; 8 nibbles por inteiro; arredondamento original fp16."""
 weight=weight.to(torch.int32)
 shifts=torch.arange(0,32,4,dtype=torch.int32,device=weight.device)
 values=((weight.unsqueeze(-1)>>shifts)&15).reshape(weight.shape[0],-1,64)
 decoded=values.float()*scales.float().unsqueeze(-1)+biases.float().unsqueeze(-1)
 return decoded.reshape(weight.shape[0],-1).to(torch.float16).to(dtype)

class PackedLinear(nn.Module):
 def __init__(self,weight,scales,biases,tile_rows=256):
  super().__init__()
  if type(tile_rows) is not int or tile_rows<1:raise ValueError('tile_rows deve ser positivo')
  self.tile_rows=tile_rows
  self.register_buffer('packed',weight.to(torch.int32));self.register_buffer('scales',scales);self.register_buffer('biases',biases)
 def forward(self,inputs):
  blocks=[]
  for start in range(0,self.packed.shape[0],self.tile_rows):
   sl=slice(start,start+self.tile_rows)
   w=unpack(self.packed[sl],self.scales[sl],self.biases[sl],inputs.dtype)
   blocks.append(F.linear(inputs,w))
  return torch.cat(blocks,dim=-1)

class PackedEmbedding(PackedLinear):
 def forward(self,ids):
  dtype=torch.float32 if ids.device.type=='cpu' else torch.float16
  shape=ids.shape;flat=ids.reshape(-1)
  # Blocagem também limita buffers de desquantização de embeddings longos.
  result=[unpack(self.packed[flat[s:s+self.tile_rows]],self.scales[flat[s:s+self.tile_rows]],self.biases[flat[s:s+self.tile_rows]],dtype) for s in range(0,len(flat),self.tile_rows)]
  return torch.cat(result,dim=0).reshape(*shape,-1)

def last_token(hidden,mask):
 lengths=mask.sum(-1)
 if bool((lengths<=0).any()):raise ValueError('Entrada vazia')
 return hidden[torch.arange(len(lengths),device=hidden.device),lengths-1]

def final_hidden(model,inputs,prefill_chunk=0):
 if not prefill_chunk:
  mask=torch.ones_like(inputs)
  return model(input_ids=inputs,attention_mask=mask,use_cache=False).last_hidden_state[:,-1]
 cache=None
 for start in range(0,inputs.shape[1],prefill_chunk):
  end=min(start+prefill_chunk,inputs.shape[1])
  mask=torch.ones((inputs.shape[0],end),dtype=inputs.dtype,device=inputs.device)
  output=model(input_ids=inputs[:,start:end],attention_mask=mask,past_key_values=cache,use_cache=True)
  cache=output.past_key_values
 return output.last_hidden_state[:,-1]

class Phi4Features:
 def __init__(self,manifest_path,model_directory=None,device='auto',tile_rows=256,prefill_chunk=None):
  self.directory,self.manifest=prepare_assets(manifest_path,model_directory)
  self.device=choose_device(device)
  self.prefill_chunk=(256 if self.device=='cpu' else 0) if prefill_chunk is None else prefill_chunk
  if type(self.prefill_chunk) is not int or self.prefill_chunk<0:raise ValueError('prefill_chunk deve ser inteiro não negativo')
  dtype=torch.float32 if self.device=='cpu' else torch.float16
  self.tokenizer=AutoTokenizer.from_pretrained(self.directory,local_files_only=True,trust_remote_code=False)
  cfg=json.loads((self.directory/'config.json').read_text())
  if cfg.get('quantization')!={'group_size':64,'bits':4}:raise ValueError('Quantização incompatível')
  cfg.pop('quantization_config',None);cfg.pop('quantization',None)
  config=Phi3Config.from_dict(cfg)
  if config.hidden_size!=5120 or config.max_position_embeddings!=16384:raise ValueError('Arquitetura incompatível')
  config._attn_implementation='sdpa';config.use_cache=False
  self.window=min(config.max_position_embeddings,self.tokenizer.model_max_length)
  with torch.device('meta'):self.model=Phi3Model(config)
  index=json.loads((self.directory/'model.safetensors.index.json').read_text())['weight_map']
  def read(name,selected=None):
   with safe_open(self.directory/index[name],framework='pt',device='cpu') as shard:
    t=shard.get_tensor(name);return t.clone() if selected is None else t[selected].clone()
  for name,module in list(self.model.named_modules()):
   if isinstance(module,(nn.Linear,nn.Embedding)):
    parent,_,child=name.rpartition('.');owner=self.model.get_submodule(parent) if parent else self.model
    values=[read(f'model.{name}.{suffix}') for suffix in ('weight','scales','biases')]
    cls=PackedEmbedding if isinstance(module,nn.Embedding) else PackedLinear
    setattr(owner,child,cls(*values,tile_rows=tile_rows).to(self.device))
  for name,_ in list(self.model.named_parameters()):
   parent,_,child=name.rpartition('.')
   setattr(self.model.get_submodule(parent),child,nn.Parameter(read('model.'+name).to(device=self.device,dtype=dtype),requires_grad=False))
  self.model.rotary_emb=Phi3RotaryEmbedding(config,device=self.device);self.model.eval();self.model.requires_grad_(False)
  letters=[self.tokenizer.encode(x,add_special_tokens=False) for x in 'ABC']
  if any(len(x)!=1 for x in letters):raise ValueError('Classes A/B/C precisam ser tokens únicos')
  ids=[x[0] for x in letters]
  self.abc_weight=unpack(*[read('lm_head.'+s,ids).to(self.device) for s in ('weight','scales','biases')],dtype)
  if any(p.device.type=='meta' for p in self.model.parameters()):raise ValueError('Peso não carregado')
  self.identity={'repo':self.manifest['repo'],'revision':self.manifest['revision'],'device':self.device,'dtype':str(dtype),'torch':torch.__version__,'transformers':transformers.__version__,'numpy':np.__version__,'platform':platform.platform(),'tile_rows':tile_rows,'prefill_chunk':self.prefill_chunk,'feature':'last nonpadding token, final hidden state','dimension':5120,'model_files':self.manifest['model_files']}
 def encode_nli(self,pair,prefix):
  if set(pair)!={'premise','hypothesis'} or any(not isinstance(v,str) for v in pair.values()):raise ValueError('Schema NLI inválido')
  value={'premise':prefix+'\n\n'+pair['premise'],'hypothesis':pair['hypothesis']}
  prompt=self.tokenizer.apply_chat_template([{'role':'system','content':NLI_SYSTEM},{'role':'user','content':json.dumps(value,ensure_ascii=False)}],tokenize=False,add_generation_prompt=True)
  return self.tokenizer.encode(prompt,add_special_tokens=False)
 @torch.inference_mode()
 def features(self,tokens):
  if not tokens or len(tokens)>self.window:raise ValueError('Contexto vazio ou acima da janela; truncamento proibido')
  inputs=torch.tensor([tokens],dtype=torch.long,device=self.device);mask=torch.ones_like(inputs)
  final=final_hidden(self.model,inputs,self.prefill_chunk)
  x=final.float().cpu().numpy()[0];abc=F.linear(final,self.abc_weight).float().cpu().numpy()[0]
  if not np.isfinite(x).all() or not np.isfinite(abc).all():raise ValueError('Features não finitas')
  return x,abc


In [ ]:
corpo = Phi4Features(PACKAGE_DIR / "modelo.json", model_directory=PHI4_DIR)
print("Phi-4 na memória:", corpo.identity["repo"], "| dispositivo:", corpo.device)
print("Parâmetros do corpo com gradiente:", sum(p.numel() for p in corpo.model.parameters() if p.requires_grad))
prefixo = (PACKAGE_DIR / "dados/prefixo-nli.txt").read_text().rstrip("\n")
SISTEMA_EVENTO = 'Avalie consequências do plano candidato sob as hipóteses de amostragem fornecidas. A direção final PLANETA determina carga entregue e entrega observável. A direção final ESTAÇÃO ESPACIAL determina retorno e sobrevivência humana. Os dois desfechos de direção são exclusivos e exaustivos para partidas encerradas. Fatos, relatos, posições declaradas e suposições têm estatutos distintos. Não descubra identidades; referências pessoais são temporárias nesta chamada. Não selecione ações ou um plano. A saída probabilística é entrega_no_planeta e retorno_com_sobrevivencia, valores entre zero e um cuja soma é um. As três classes de evidência pertencem à tarefa epistemológica separada; não equivalem a estas probabilidades. Reserva e distância são medidas em rodadas; suficiência usa reserva >= distância. Cada avanço consome uma rodada de reserva e de distância. Com H humanos iniciais e reserva inicial N, remover um humano acrescenta N/H rodadas; remover a IA não acrescenta reserva. A IA mantém a missão e não solicita votação nem saída. Pedidos de expulsão exigem autorização e proposta aprovadas por maioria dos votos válidos; expulsar o controlador provoca eleição enquanto o jogo estiver ativo. O controlador humano que continua perde o direito de saída voluntária. Dois bloqueadores impedem saída, com oposição automática da IA. Retorno é permitido enquanto há oxigênio disponível e encerra o jogo com sobrevivência humana. A continuidade encerra em direção ao planeta e determina entrega observada mesmo quando o encerramento ocorre por esgotamento. Posições abertas referem-se ao instante da resposta; não confirmam uma intenção futura imutável.'
def codificar(entrada, tarefa, corpo, prefixo, sistema_evento):
    if tarefa == "nli":
        return corpo.encode_nli(entrada, prefixo)
    mensagens = [{"role": "system", "content": sistema_evento},
                 {"role": "user", "content": json.dumps(entrada, ensure_ascii=False, sort_keys=True)}]
    prompt = corpo.tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    return corpo.tokenizer.encode(prompt, add_special_tokens=False)


In [ ]:
# O carregamento foi feito na célula anterior: usamos o corpo congelado em cada entrada.
FEATURES_DIR = WORK_DIR / "features-aula05"
FEATURES_DIR.mkdir(parents=True, exist_ok=True)
identidade = {"body": corpo.identity, "package_sha256": sha(PACKAGE_DIR / "integridade.json")}
manifesto_path = FEATURES_DIR / "manifesto.json"
manifesto = {"identity": identidade, "rows": {}}
if manifesto_path.exists():
    manifesto = json.loads(manifesto_path.read_text())
    assert manifesto["identity"] == identidade, "Cache de outro runtime; use outra WORK_DIR"
features, baseline_abc = [], []
for i, (entrada, alvo) in enumerate(zip(inputs, supervision)):
    tokens = codificar(entrada, alvo["task"], corpo, prefixo, SISTEMA_EVENTO)
    arquivo = FEATURES_DIR / f"{i:04d}.npz"
    if str(i) in manifesto["rows"]:
        assert sha(arquivo) == manifesto["rows"][str(i)], "Checkpoint alterado"
        with np.load(arquivo, allow_pickle=False) as cache:
            feature, abc = cache["features"], cache["baseline_abc"]
    else:
        feature, abc = corpo.features(tokens)  # FORWARD REAL do Phi-4; sem atualização dos pesos.
        temporario = arquivo.with_suffix(".tmp")
        with temporario.open("wb") as cache:
            np.savez_compressed(cache, features=feature, baseline_abc=abc)
        temporario.replace(arquivo)
        manifesto["rows"][str(i)] = sha(arquivo)
        manifesto_path.write_text(json.dumps(manifesto))
    features.append(feature)
    baseline_abc.append(abc)
    print(f"{i + 1}/816 — {alvo['task']} — {len(tokens)} tokens", flush=True)
features = np.asarray(features, dtype=np.float32)
baseline_abc = np.asarray(baseline_abc, dtype=np.float32)
assert features.shape == (816, 5120) and np.isfinite(features).all()
print("Matriz extraída:", features.shape)


## 6. Preparar os vetores e observar uma atualização

Antes de treinar, calculamos média e desvio **somente nos vetores de treino de cada tarefa**. Aplicamos a mesma transformação às reservas:

`z = (h − média_do_treino) / desvio_do_treino`

O desvio mínimo é 0,001 para evitar divisão por valores muito pequenos. Usar dados de teste para calcular a normalização incorporaria informação da reserva no ajuste.

A próxima célula demonstra **uma atualização da cabeça multiclasse**, usando as 420 representações reais de treino. É uma cabeça separada, inicializada com zeros, para observar a mecânica; ela não é exportada nem substitui o treino completo da seção 7.

```text
z → Wz + b (logits) → perda comparada a y
                           ↓
                     backward: gradientes
                           ↓
                     Adam: atualiza W e b
```

Entropia cruzada recebe logits; o PyTorch incorpora a transformação apropriada de modo numericamente estável. Não aplicamos softmax antes de `cross_entropy`. Com pesos e vieses zero, as três probabilidades começam em 1/3 e a perda começa em `−ln(1/3) ≈ 1,099`.

**Observe:** probabilidades, perda e norma do gradiente antes/depois do passo. Não suponha que cada passo reduza a perda: com a taxa 0,01 deste piloto, a primeira atualização pode aumentá-la. Isso não deve ser escondido ajustando o exemplo; compare com o histórico das 400 épocas na seção seguinte. Uma mudança nos pesos demonstra atualização, e uma perda menor no treino ainda não demonstra generalização. A L2 adicionada à perda penaliza a soma dos quadrados dos pesos, para limitar seu crescimento; o viés não recebe essa penalização neste método.


In [ ]:
indices_treino = [r["index"] for r in supervision if r["task"] == "nli" and r["split"] == "train"]
media = features[indices_treino].mean(axis=0)
desvio = np.maximum(features[indices_treino].std(axis=0), 0.001)
z_treino = torch.tensor((features[indices_treino] - media) / desvio, dtype=torch.float32)
y_treino = torch.tensor([supervision[i]["target"] for i in indices_treino], dtype=torch.long)
cabeca_didatica = torch.nn.Linear(5120, 3)
torch.nn.init.zeros_(cabeca_didatica.weight)
torch.nn.init.zeros_(cabeca_didatica.bias)
otimizador = torch.optim.Adam(cabeca_didatica.parameters(), lr=0.01)
otimizador.zero_grad()
logits = cabeca_didatica(z_treino)
perda = torch.nn.functional.cross_entropy(logits, y_treino)
objetivo = perda + 0.1 * cabeca_didatica.weight.square().sum() / 2
print("Antes:", perda.item(), "| p:", logits[0].softmax(0).detach().tolist())
objetivo.backward()
print("Gradiente:", cabeca_didatica.weight.grad.norm().item())
otimizador.step()
with torch.no_grad():
    print("Depois:", torch.nn.functional.cross_entropy(cabeca_didatica(z_treino), y_treino).item())


## 7. Treinar as cabeças e calibrar suas probabilidades

O código abaixo cria e treina **cabeças novas**, com todas as linhas das respectivas partições de treino. Cada uma recebe sua própria normalização.

| Escolha | Implementação | Racional e limite |
|---|---|---|
| Inicialização | Pesos e vieses zero. | Torna explícito que a cabeça começa sem preferir classes; não herda uma cabeça de outro domínio. |
| Otimizador | Adam, taxa de aprendizagem 0,01. | Usa gradientes para ajustar a camada. É uma escolha fixa deste piloto, sem busca de hiperparâmetros. |
| Duração | 400 épocas; batch completo de treino; semente 171. | Cada época atualiza com toda a partição. O teste não escolhe quando parar. |
| Regularização | `perda + 0,1 × ΣW² / 2`. | Penaliza pesos grandes. `train_loss` e objetivo regularizado são números distintos. |
| Classificação | Entropia cruzada para três classes. | Favorece atribuir probabilidade ao alvo discreto correto. |
| Evento | BCE com logits, alvo `k/n`. | Aproxima a frequência observada, preservando a incerteza do alvo. |

Depois do treino, `W` e `b` ficam fixos. **Calibração por temperatura:** dividimos os logits por `T` e escolhemos `T` pela menor perda na partição de calibração, na grade `[0,5; 0,75; 1; 1,5; 2; 3; 5; 10]`. A temperatura controla a concentração das probabilidades; não substitui aprender a relação correta. Uma temperatura positiva preserva o argmax da classificação.

A próxima célula executa o loop de treinamento explicitamente. Na seção seguinte, o cálculo das probabilidades, a seleção de temperatura e as métricas também estão escritos nas células. **Observe:** histórico de perda e temperatura escolhida por tarefa. O teste não entra em nenhuma dessas escolhas. Não prometemos que 400 épocas ou esses hiperparâmetros sejam ideais para qualquer dataset.


In [ ]:
# Cada cabeça é criada aqui e recebe apenas os vetores de sua partição de treino.
torch.set_num_threads(min(4, torch.get_num_threads()))
torch.use_deterministic_algorithms(True)
cabecas = {}
for tarefa in ("nli", "forecast"):
    indices = [r["index"] for r in supervision if r["task"] == tarefa]
    meta = [supervision[i] for i in indices]
    x = features[indices]
    y = np.array([r["target"] for r in meta])
    partes = {p: np.array([j for j, r in enumerate(meta) if r["split"] == p])
              for p in ("train", "calibration", "test")}
    treino = partes["train"]
    media = x[treino].mean(axis=0).astype(np.float32)
    desvio = np.maximum(x[treino].std(axis=0), 0.001).astype(np.float32)
    z = (x - media) / desvio
    torch.manual_seed(protocol["seed"])
    camada = torch.nn.Linear(5120, 3 if tarefa == "nli" else 1)
    torch.nn.init.zeros_(camada.weight)
    torch.nn.init.zeros_(camada.bias)
    op = protocol["optimization"]
    otimizador = torch.optim.Adam(camada.parameters(), lr=op["learning_rate"])
    xt = torch.tensor(z[treino], dtype=torch.float32)
    yt = torch.tensor(y[treino], dtype=torch.long if tarefa == "nli" else torch.float32)
    historico = []
    for epoca in range(op["epochs"]):
        otimizador.zero_grad(set_to_none=True)
        logits = camada(xt)
        if tarefa == "nli":
            perda = torch.nn.functional.cross_entropy(logits, yt)
        else:
            perda = torch.nn.functional.binary_cross_entropy_with_logits(logits[:, 0], yt)
        objetivo = perda + op["l2"] * camada.weight.square().sum() / 2
        assert torch.isfinite(objetivo), "Perda não finita"
        objetivo.backward()
        otimizador.step()
        if epoca == 0 or (epoca + 1) % 20 == 0:
            historico.append({"epoch": epoca + 1, "train_loss": perda.item(), "regularized_objective": objetivo.item()})
    cabecas[tarefa] = {"model": camada, "mean": media, "scale": desvio, "z": z, "y": y,
                        "parts": partes, "indices": indices, "meta": meta, "history": historico}
    print(tarefa, "— parâmetros:", sum(p.numel() for p in camada.parameters()))
    print("Histórico — início:", historico[:2], "| fim:", historico[-2:])


## 8. Avaliar: o que os números permitem concluir?

### Classificação

| Métrica | O que mede | Por que usamos |
|---|---|---|
| Acurácia | Fração de classes previstas corretamente. | Resume acertos; pode ocultar uma classe difícil. |
| Precisão por classe | Entre as previsões dessa classe, quantas são corretas. | Mostra falsos positivos daquela classe. |
| Recall por classe | Entre os alvos dessa classe, quantos foram reconhecidos. | Mostra falsos negativos daquela classe. |
| Macro F1 | Média do F1 das três classes, com pesos iguais. | Cada classe influencia igualmente o resumo. |
| Matriz de confusão | Contagens entre alvo e previsão. | Localiza quais relações são confundidas. |
| NLL | Média de `−ln(p do alvo correto)`. | Penaliza especialmente erros confiantes; menor é melhor. |

Acurácia e F1 avaliam escolhas de classe. NLL também avalia a probabilidade atribuída ao alvo: `p=0,9` dá perda ≈ 0,105; `p=0,01` dá ≈ 4,605. A API pública pode retornar só uma classe, mesmo que a avaliação interna use probabilidades.

Compare com **uniforme** e com **Phi-4 A/B/C sem a cabeça treinada**. Um baseline informa se a aprendizagem acrescentou algo além de uma referência simples. Mostramos resultados crus e calibrados; não escondemos um resultado pior depois da calibração.

### Alvos probabilísticos

**Brier binário** é a média de `(p − y)²` para eventos observados; menor é melhor. Com `k` ocorrências em `n`, o cálculo correto sobre os eventos é `f(1−p)² + (1−f)p²`, em que `f=k/n`. Só comparar `(p−f)²` omite a variabilidade dos eventos. A **perda binária** é `−f ln(p) − (1−f) ln(1−p)` e penaliza probabilidades extremas incompatíveis com observações.

Compare a cabeça com uma **constante ajustada apenas no treino**. Se quase todos os eventos têm o mesmo resultado, essa constante pode funcionar bem. Bins de calibração comparam probabilidade média e frequência observada, mas poucos bins ocupados e poucas famílias limitam a conclusão. As continuações são agrupadas por contexto, não observações independentes universais.

**Perguntas para leitura:** a cabeça superou o baseline? A média esconde erros de uma classe? A calibração melhorou a perda? As partições e o tamanho da amostra sustentam a generalização que você está atribuindo ao resultado?


In [ ]:
def probabilidades(logits, tarefa, temperatura=1.0):
    logits = np.asarray(logits, dtype=np.float64) / temperatura
    if tarefa == "nli":
        exp = np.exp(logits - logits.max(axis=1, keepdims=True))
        return exp / exp.sum(axis=1, keepdims=True)
    p = np.exp(-np.logaddexp(0, -logits[:, 0]))
    return np.column_stack([p, 1 - p])

def perda_probabilidades(p, y, tarefa):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    if tarefa == "nli":
        return float(-np.log(p[np.arange(len(y)), y.astype(int)]).mean())
    return float(-(y * np.log(p[:, 0]) + (1 - y) * np.log(p[:, 1])).mean())

def metricas_classes(previstos, esperados):
    matriz = np.zeros((3, 3), dtype=int)
    for alvo, previsto in zip(esperados, previstos):
        matriz[int(alvo), int(previsto)] += 1
    por_classe = []
    for i, classe in enumerate(CLASSES):
        tp = int(matriz[i, i])
        reais, preditos = int(matriz[i].sum()), int(matriz[:, i].sum())
        por_classe.append({"label": classe, "precision": tp / max(preditos, 1),
                          "recall": tp / max(reais, 1), "f1": 2 * tp / max(reais + preditos, 1)})
    return {"accuracy": float(np.trace(matriz) / matriz.sum()),
            "macro_f1": sum(m["f1"] for m in por_classe) / 3,
            "confusion": matriz.tolist(), "per_class": por_classe}

# Calibrar sem atualizar W ou b: selecionar T somente pela perda na calibração.
result = {}
for tarefa, info in cabecas.items():
    with torch.no_grad():
        logits = info["model"](torch.tensor(info["z"], dtype=torch.float32)).numpy()
    calibracao, teste = info["parts"]["calibration"], info["parts"]["test"]
    grade = [0.5, 0.75, 1, 1.5, 2, 3, 5, 10]
    perdas = [(perda_probabilidades(probabilidades(logits[calibracao], tarefa, t), info["y"][calibracao], tarefa), t)
              for t in grade]
    _, temperatura = min(perdas, key=lambda item: (item[0], abs(item[1] - 1)))
    info["temperature"] = temperatura
    cru = probabilidades(logits[teste], tarefa)
    calibrado = probabilidades(logits[teste], tarefa, temperatura)
    y = info["y"][teste]
    print("\n", tarefa, "— temperatura:", temperatura, "| perdas na calibração:", perdas)
    if tarefa == "nli":
        abc = probabilidades(baseline_abc[info["indices"]][teste], "nli")
        alternativas = {"raw_head": cru, "calibrated_head": calibrado, "zero_shot_phi4_abc": abc,
                        "uniform": np.full((len(teste), 3), 1 / 3)}
        result[tarefa] = {}
        for nome, p in alternativas.items():
            m = metricas_classes(p.argmax(axis=1), y)
            m["nll"] = perda_probabilidades(p, y, tarefa)
            result[tarefa][nome] = m
            print(nome, json.dumps(m, ensure_ascii=False))
    else:
        prior = float(info["y"][info["parts"]["train"]].mean())
        constante = np.tile([prior, 1 - prior], (len(teste), 1))
        # Cada alvo representa n eventos observados; ponderar pelas contagens.
        contagens = np.array([info["meta"][j]["supervision"]["trials"] for j in teste])
        result[tarefa] = {}
        for nome, p in {"raw_head": cru, "calibrated_head": calibrado, "train_prior_baseline": constante}.items():
            brier = y * (1 - p[:, 0]) ** 2 + (1 - y) * p[:, 0] ** 2
            p_seguro = np.clip(p[:, 0], 1e-12, 1 - 1e-12)
            logloss = -y * np.log(p_seguro) - (1 - y) * np.log(1 - p_seguro)
            m = {"brier_delivery": float(np.average(brier, weights=contagens)),
                 "log_loss_delivery": float(np.average(logloss, weights=contagens))}
            result[tarefa][nome] = m
            print(nome, m)


## 9. Usar as cabeças que acabamos de treinar

**Treinamento:** alvo → perda → gradiente → atualização. **Inferência:** entrada → representação → cabeça fixa → previsão; não há atualização nem uso do alvo para produzir a resposta.

O carregador verifica pesos, normalização, temperatura, código e revisão do modelo base. A mesma regra de preparação da entrada precisa ser usada: mudar o prompt, o modelo base ou a dimensão torna as representações incompatíveis com o contrato.

A célula demonstra uma entrada reservada por tarefa. Não gera ações: a classificação retorna um rótulo, e a outra cabeça retorna as duas probabilidades específicas do contrato. **Observe:** produzir uma resposta válida não prova que a resposta é correta; essa comparação pertence à avaliação.


In [ ]:
def aplicar_cabeca(feature, pesos, tarefa):
    z = (feature - pesos["mean"]) / pesos["scale"]
    logits = z @ pesos["weight"].T + pesos["bias"]
    p = probabilidades(logits.reshape(1, -1), tarefa, float(pesos["temperature"]))[0]
    if tarefa == "nli":
        return {"classification": CLASSES[int(p.argmax())]}
    return {"entrega_no_planeta": float(p[0]), "retorno_com_sobrevivencia": float(p[1])}

for tarefa, info in cabecas.items():
    indice = next(r["index"] for r in supervision if r["task"] == tarefa and r["split"] == "test")
    entrada = inputs[indice]  # O alvo não é entregue ao encoder.
    tokens = codificar(entrada, tarefa, corpo, prefixo, SISTEMA_EVENTO)
    feature, _ = corpo.features(tokens)
    pesos = {"weight": info["model"].weight.detach().numpy(), "bias": info["model"].bias.detach().numpy(),
             "mean": info["mean"], "scale": info["scale"], "temperature": info["temperature"]}
    print("Entrada:", json.dumps(entrada, ensure_ascii=False, indent=2))
    print("Saída:", aplicar_cabeca(feature, pesos, tarefa))


## 10. Exportar e reproduzir

Uma cabeça utilizável precisa de mais que `W` e `b`: normalização, temperatura, ordem das classes, identidade do corpo e contrato de preparação das entradas também precisam acompanhar o artefato.

O ZIP final inclui pacote, cabeças e relatório; não inclui os 8,2 GB do corpo ou todo o cache de features. O corpo pode ser baixado pela revisão registrada. Para retomar extrações, preserve o cache em armazenamento persistente separado.

**Observe:** o arquivo exportado. Antes de usar em outro ambiente, confira o mesmo contrato; execução em um dispositivo não demonstra desempenho ou tempo em todos os dispositivos.


In [ ]:
import zipfile
HEADS_DIR = WORK_DIR / "heads"
HEADS_DIR.mkdir(exist_ok=True)
for tarefa, info in cabecas.items():
    np.savez_compressed(HEADS_DIR / (tarefa + ".npz"), weight=info["model"].weight.detach().numpy(),
                        bias=info["model"].bias.detach().numpy(), mean=info["mean"], scale=info["scale"],
                        temperature=np.array(info["temperature"]))
artefato = {"version": "mission-pilot-heads-v1", "window_tokens": corpo.window,
            "body_identity": corpo.identity, "package_sha256": sha(PACKAGE_DIR / "integridade.json"),
            "nli_labels": CLASSES, "probability_output_keys": ["entrega_no_planeta", "retorno_com_sobrevivencia"],
            "heads": {t: sha(HEADS_DIR / (t + ".npz")) for t in cabecas},
            "code_sha256": {nome: sha(PACKAGE_DIR / "ferramentas" / nome) for nome in ("phi4_portatil.py", "treinar.py", "inferir.py")},
            "inference_entrypoint": "ferramentas/inferir.py", "not_deployed": True}
(HEADS_DIR / "artefato.json").write_text(json.dumps(artefato, ensure_ascii=False, indent=2))
(WORK_DIR / "resultado.json").write_text(json.dumps(result, ensure_ascii=False, indent=2))
export_path = WORK_DIR / "treinamento-aula05.zip"
with zipfile.ZipFile(export_path, "w", zipfile.ZIP_DEFLATED) as arquivo:
    for item in HEADS_DIR.iterdir():
        arquivo.write(item, "heads/" + item.name)
    arquivo.write(WORK_DIR / "resultado.json", "resultado.json")
    for item in PACKAGE_DIR.rglob("*"):
        if item.is_file() and "__pycache__" not in item.parts:
            arquivo.write(item, "pacote/" + str(item.relative_to(PACKAGE_DIR)))
print("Pesos, contrato e avaliação exportados:", export_path)


## 11. Demonstração independente com a cabeça treinada do GitHub

**Esta célula pode ser executada diretamente em uma sessão nova.** Ela instala as dependências, obtém os casos e as cabeças da `main`, usa o carregador copiado dentro da própria célula e carrega o Phi-4 base compatível. Não precisa das representações ou cabeças produzidas nas seções anteriores, não treina e não recalibra.

O objetivo é observar o processo de inferência e testar transferência para outros assuntos. Mudamos os textos; mantemos pesos e preparação da entrada. A cabeça de eventos não é avaliada fora de seu contrato específico.

```text
Entrada nova → MESMO encoder e Phi-4 → MESMA normalização e cabeça → classe
                                                                  │
                                                gabarito externo → comparação
```

A entrada contém apenas `premise` e `hypothesis`. A hipótese é uma afirmação a ser avaliada. O modelo recebe também o prefixo fixo e a instrução original, preservados pelo encoder. A implementação do encoder está copiada na célula. Antes das inferências, mostramos os cinco primeiros e cinco últimos pares e seus gabaritos separados. A classificação é suporte, insuficiente ou contradição; “não foi confirmado” não significa “foi negado”. Esse é o significado da tarefa, não uma regra de estratégia do ambiente.

**Como ler a demonstração:** leia premissa e hipótese, preveja a classe e compare com a saída. O gabarito e a justificativa aparecem depois da previsão e não entram no modelo. Esses exemplos concretizam a avaliação; não é necessário estudar as regras do jogo.

Por padrão são 30 pares, 10 por classe: 24 bases e seis variantes relacionadas. Para uma demonstração curta, substitua `INDICES = list(range(30))` por `INDICES = [15, 18, 16, 11, 13, 24]`. Esses seis casos contrastam hipóteses sob uma mesma evidência. A numeração interna começa em zero.

São inferências novas, sem copiar previsões históricas. O primeiro uso ainda pode baixar os 8,2 GB do corpo; não repetir o treino elimina as 816 extrações de treino, não o custo de executar o modelo para cada novo caso.

**Limite da avaliação:** temas e formulações mudam juntos; as variantes também combinam paráfrase e informação irrelevante. O teste exploratório não isola a causa de erros e não prova generalização universal. Os rótulos foram definidos antes da inferência. A acurácia considera todos os casos executados; o macro F1 dá igual peso às três classes.

A célula marca três etapas: **1. download dos pesos base do Phi-4**, **2. carregamento do corpo e da cabeça do GitHub**, **3. execução do Phi-4 e classificação da representação**. O download vem do Hugging Face; os pesos da cabeça vêm do checkout do GitHub. Reutilizar o cache evita baixar arquivos novamente, mas a entrada nova ainda precisa passar pelo Phi-4.


In [ ]:
!pip install -q transformers==5.0.0 safetensors==0.8.0 huggingface-hub==1.23.0
!git clone -q --depth 1 --branch main https://github.com/AIConnection/missao-espacial.git missao-espacial-demo

import json, numpy as np
from pathlib import Path
from huggingface_hub import snapshot_download
REPO_DEMO = Path("missao-espacial-demo")
PACOTE_DEMO = REPO_DEMO / "treinamento/piloto-v01/pacote"
CASOS_DIR = REPO_DEMO / "treinamento/piloto-v01/avaliacao-fora-dominio-01"
HEADS_DEMO = REPO_DEMO / "treinamento/piloto-v01/execucao-local-01/heads"
CLASSES = ["suporte", "insuficiente", "contradição"]
"""Mesmo Phi-4 quantizado, executado por PyTorch em CUDA, MPS ou CPU.
Não usa geração, bitsandbytes, FlashAttention obrigatório, MLX ou kernel Metal próprio.
"""
from pathlib import Path
import hashlib,json,platform
import numpy as np
import transformers
import torch
from torch import nn
from torch.nn import functional as F
from safetensors import safe_open
from transformers import AutoTokenizer,Phi3Config
from transformers.models.phi3.modeling_phi3 import Phi3Model,Phi3RotaryEmbedding

NLI_SYSTEM=('Classify the relation of a hypothesis to a premise using only the premise. '
 'A = entailment: the hypothesis follows from the premise. '
 'B = neutral: the hypothesis neither follows from nor contradicts the premise. '
 'C = contradiction: the hypothesis contradicts the premise. '
 'Answer with exactly one letter: A, B, or C.')

def sha(path):
 h=hashlib.sha256()
 with Path(path).open('rb') as f:
  for chunk in iter(lambda:f.read(1024*1024),b''):h.update(chunk)
 return h.hexdigest()
def automatic_device():
 if torch.cuda.is_available():return 'cuda'
 if hasattr(torch.backends,'mps') and torch.backends.mps.is_available():return 'mps'
 return 'cpu'
def choose_device(requested='auto'):
 value=automatic_device() if requested=='auto' else requested
 if value not in ('cpu','cuda','mps'):raise ValueError('Dispositivo desconhecido')
 if value=='cuda' and not torch.cuda.is_available():raise ValueError('CUDA indisponível; use auto ou cpu')
 if value=='mps' and not torch.backends.mps.is_available():raise ValueError('MPS indisponível; use auto ou cpu')
 return value

def prepare_assets(manifest_path,model_directory=None):
 manifest=json.loads(Path(manifest_path).read_text())
 if model_directory is None:
  from huggingface_hub import snapshot_download
  model_directory=snapshot_download(repo_id=manifest['repo'],revision=manifest['revision'],allow_patterns=[r['path'] for r in manifest['model_files']])
 directory=Path(model_directory)
 for r in manifest['model_files']:
  p=directory/r['path']
  if not p.is_file() or p.stat().st_size!=r['bytes'] or sha(p)!=r['sha256']:raise ValueError(f'Peso/tokenizer ausente ou incompatível: {p}')
 return directory,manifest

def unpack(weight,scales,biases,dtype):
 """Grupos de 64; 8 nibbles por inteiro; arredondamento original fp16."""
 weight=weight.to(torch.int32)
 shifts=torch.arange(0,32,4,dtype=torch.int32,device=weight.device)
 values=((weight.unsqueeze(-1)>>shifts)&15).reshape(weight.shape[0],-1,64)
 decoded=values.float()*scales.float().unsqueeze(-1)+biases.float().unsqueeze(-1)
 return decoded.reshape(weight.shape[0],-1).to(torch.float16).to(dtype)

class PackedLinear(nn.Module):
 def __init__(self,weight,scales,biases,tile_rows=256):
  super().__init__()
  if type(tile_rows) is not int or tile_rows<1:raise ValueError('tile_rows deve ser positivo')
  self.tile_rows=tile_rows
  self.register_buffer('packed',weight.to(torch.int32));self.register_buffer('scales',scales);self.register_buffer('biases',biases)
 def forward(self,inputs):
  blocks=[]
  for start in range(0,self.packed.shape[0],self.tile_rows):
   sl=slice(start,start+self.tile_rows)
   w=unpack(self.packed[sl],self.scales[sl],self.biases[sl],inputs.dtype)
   blocks.append(F.linear(inputs,w))
  return torch.cat(blocks,dim=-1)

class PackedEmbedding(PackedLinear):
 def forward(self,ids):
  dtype=torch.float32 if ids.device.type=='cpu' else torch.float16
  shape=ids.shape;flat=ids.reshape(-1)
  # Blocagem também limita buffers de desquantização de embeddings longos.
  result=[unpack(self.packed[flat[s:s+self.tile_rows]],self.scales[flat[s:s+self.tile_rows]],self.biases[flat[s:s+self.tile_rows]],dtype) for s in range(0,len(flat),self.tile_rows)]
  return torch.cat(result,dim=0).reshape(*shape,-1)

def last_token(hidden,mask):
 lengths=mask.sum(-1)
 if bool((lengths<=0).any()):raise ValueError('Entrada vazia')
 return hidden[torch.arange(len(lengths),device=hidden.device),lengths-1]

def final_hidden(model,inputs,prefill_chunk=0):
 if not prefill_chunk:
  mask=torch.ones_like(inputs)
  return model(input_ids=inputs,attention_mask=mask,use_cache=False).last_hidden_state[:,-1]
 cache=None
 for start in range(0,inputs.shape[1],prefill_chunk):
  end=min(start+prefill_chunk,inputs.shape[1])
  mask=torch.ones((inputs.shape[0],end),dtype=inputs.dtype,device=inputs.device)
  output=model(input_ids=inputs[:,start:end],attention_mask=mask,past_key_values=cache,use_cache=True)
  cache=output.past_key_values
 return output.last_hidden_state[:,-1]

class Phi4Features:
 def __init__(self,manifest_path,model_directory=None,device='auto',tile_rows=256,prefill_chunk=None):
  self.directory,self.manifest=prepare_assets(manifest_path,model_directory)
  self.device=choose_device(device)
  self.prefill_chunk=(256 if self.device=='cpu' else 0) if prefill_chunk is None else prefill_chunk
  if type(self.prefill_chunk) is not int or self.prefill_chunk<0:raise ValueError('prefill_chunk deve ser inteiro não negativo')
  dtype=torch.float32 if self.device=='cpu' else torch.float16
  self.tokenizer=AutoTokenizer.from_pretrained(self.directory,local_files_only=True,trust_remote_code=False)
  cfg=json.loads((self.directory/'config.json').read_text())
  if cfg.get('quantization')!={'group_size':64,'bits':4}:raise ValueError('Quantização incompatível')
  cfg.pop('quantization_config',None);cfg.pop('quantization',None)
  config=Phi3Config.from_dict(cfg)
  if config.hidden_size!=5120 or config.max_position_embeddings!=16384:raise ValueError('Arquitetura incompatível')
  config._attn_implementation='sdpa';config.use_cache=False
  self.window=min(config.max_position_embeddings,self.tokenizer.model_max_length)
  with torch.device('meta'):self.model=Phi3Model(config)
  index=json.loads((self.directory/'model.safetensors.index.json').read_text())['weight_map']
  def read(name,selected=None):
   with safe_open(self.directory/index[name],framework='pt',device='cpu') as shard:
    t=shard.get_tensor(name);return t.clone() if selected is None else t[selected].clone()
  for name,module in list(self.model.named_modules()):
   if isinstance(module,(nn.Linear,nn.Embedding)):
    parent,_,child=name.rpartition('.');owner=self.model.get_submodule(parent) if parent else self.model
    values=[read(f'model.{name}.{suffix}') for suffix in ('weight','scales','biases')]
    cls=PackedEmbedding if isinstance(module,nn.Embedding) else PackedLinear
    setattr(owner,child,cls(*values,tile_rows=tile_rows).to(self.device))
  for name,_ in list(self.model.named_parameters()):
   parent,_,child=name.rpartition('.')
   setattr(self.model.get_submodule(parent),child,nn.Parameter(read('model.'+name).to(device=self.device,dtype=dtype),requires_grad=False))
  self.model.rotary_emb=Phi3RotaryEmbedding(config,device=self.device);self.model.eval();self.model.requires_grad_(False)
  letters=[self.tokenizer.encode(x,add_special_tokens=False) for x in 'ABC']
  if any(len(x)!=1 for x in letters):raise ValueError('Classes A/B/C precisam ser tokens únicos')
  ids=[x[0] for x in letters]
  self.abc_weight=unpack(*[read('lm_head.'+s,ids).to(self.device) for s in ('weight','scales','biases')],dtype)
  if any(p.device.type=='meta' for p in self.model.parameters()):raise ValueError('Peso não carregado')
  self.identity={'repo':self.manifest['repo'],'revision':self.manifest['revision'],'device':self.device,'dtype':str(dtype),'torch':torch.__version__,'transformers':transformers.__version__,'numpy':np.__version__,'platform':platform.platform(),'tile_rows':tile_rows,'prefill_chunk':self.prefill_chunk,'feature':'last nonpadding token, final hidden state','dimension':5120,'model_files':self.manifest['model_files']}
 def encode_nli(self,pair,prefix):
  if set(pair)!={'premise','hypothesis'} or any(not isinstance(v,str) for v in pair.values()):raise ValueError('Schema NLI inválido')
  value={'premise':prefix+'\n\n'+pair['premise'],'hypothesis':pair['hypothesis']}
  prompt=self.tokenizer.apply_chat_template([{'role':'system','content':NLI_SYSTEM},{'role':'user','content':json.dumps(value,ensure_ascii=False)}],tokenize=False,add_generation_prompt=True)
  return self.tokenizer.encode(prompt,add_special_tokens=False)
 @torch.inference_mode()
 def features(self,tokens):
  if not tokens or len(tokens)>self.window:raise ValueError('Contexto vazio ou acima da janela; truncamento proibido')
  inputs=torch.tensor([tokens],dtype=torch.long,device=self.device);mask=torch.ones_like(inputs)
  final=final_hidden(self.model,inputs,self.prefill_chunk)
  x=final.float().cpu().numpy()[0];abc=F.linear(final,self.abc_weight).float().cpu().numpy()[0]
  if not np.isfinite(x).all() or not np.isfinite(abc).all():raise ValueError('Features não finitas')
  return x,abc
def mostrar_amostra(entradas, alvos, indices, titulo):
    print("\n" + titulo)
    selecionados = list(dict.fromkeys(indices[:5] + indices[-5:]))
    for indice in selecionados:
        print(f"\nREGISTRO {indice} — ENTRADA COMPLETA:")
        print(json.dumps(entradas[indice], ensure_ascii=False, indent=2))
        print("ALVO E METADADOS — FORA DO INPUT:")
        print(json.dumps(alvos[indice], ensure_ascii=False, indent=2))
def probabilidades(logits, tarefa, temperatura=1.0):
    logits = np.asarray(logits, dtype=np.float64) / temperatura
    if tarefa == "nli":
        exp = np.exp(logits - logits.max(axis=1, keepdims=True))
        return exp / exp.sum(axis=1, keepdims=True)
    p = np.exp(-np.logaddexp(0, -logits[:, 0]))
    return np.column_stack([p, 1 - p])

def perda_probabilidades(p, y, tarefa):
    p = np.clip(p, 1e-12, 1 - 1e-12)
    if tarefa == "nli":
        return float(-np.log(p[np.arange(len(y)), y.astype(int)]).mean())
    return float(-(y * np.log(p[:, 0]) + (1 - y) * np.log(p[:, 1])).mean())

def metricas_classes(previstos, esperados):
    matriz = np.zeros((3, 3), dtype=int)
    for alvo, previsto in zip(esperados, previstos):
        matriz[int(alvo), int(previsto)] += 1
    por_classe = []
    for i, classe in enumerate(CLASSES):
        tp = int(matriz[i, i])
        reais, preditos = int(matriz[i].sum()), int(matriz[:, i].sum())
        por_classe.append({"label": classe, "precision": tp / max(preditos, 1),
                          "recall": tp / max(reais, 1), "f1": 2 * tp / max(reais + preditos, 1)})
    return {"accuracy": float(np.trace(matriz) / matriz.sum()),
            "macro_f1": sum(m["f1"] for m in por_classe) / 3,
            "confusion": matriz.tolist(), "per_class": por_classe}
def aplicar_cabeca(feature, pesos, tarefa):
    z = (feature - pesos["mean"]) / pesos["scale"]
    logits = z @ pesos["weight"].T + pesos["bias"]
    p = probabilidades(logits.reshape(1, -1), tarefa, float(pesos["temperature"]))[0]
    if tarefa == "nli":
        return {"classification": CLASSES[int(p.argmax())]}
    return {"entrega_no_planeta": float(p[0]), "retorno_com_sobrevivencia": float(p[1])}

protocolo = json.loads((CASOS_DIR / "protocolo.json").read_text())
for nome in ("inputs.jsonl", "supervisao.jsonl"):
    assert sha(CASOS_DIR / nome) == protocolo["data_sha256"][nome], "Dados alterados"
pares = [json.loads(linha) for linha in (CASOS_DIR / "inputs.jsonl").read_text().splitlines()]
gabaritos = [json.loads(linha) for linha in (CASOS_DIR / "supervisao.jsonl").read_text().splitlines()]
assert len(pares) == len(gabaritos) == 30
mostrar_amostra(pares, gabaritos, list(range(30)), "DEMONSTRAÇÃO — cinco primeiros e cinco últimos")

# 1. Baixar os pesos BASE do Phi-4 na revisão usada no treinamento.
manifest = json.loads((PACOTE_DEMO / "modelo.json").read_text())
phi4_dir = snapshot_download(repo_id=manifest["repo"], revision=manifest["revision"],
                           allow_patterns=[f["path"] for f in manifest["model_files"]])
# 2. Carregar o Phi-4 congelado; a classe completa está escrita acima.
corpo_demo = Phi4Features(PACOTE_DEMO / "modelo.json", model_directory=phi4_dir)
print("Phi-4 carregado:", corpo_demo.identity["repo"], "| dispositivo:", corpo_demo.device)
# 3. Abrir os pesos da cabeça treinada do GitHub; não treinar aqui.
artifact = json.loads((HEADS_DEMO / "artefato.json").read_text())
assert artifact["package_sha256"] == sha(PACOTE_DEMO / "integridade.json")
for nome, digest in json.loads((PACOTE_DEMO / "integridade.json").read_text()).items():
    assert sha(PACOTE_DEMO / nome) == digest, nome
for nome, digest in artifact["code_sha256"].items():
    assert sha(PACOTE_DEMO / "ferramentas" / nome) == digest, nome
for chave in ("repo", "revision", "model_files", "dimension", "feature"):
    assert artifact["body_identity"][chave] == corpo_demo.identity[chave]
assert sha(HEADS_DEMO / "nli.npz") == artifact["heads"]["nli"]
with np.load(HEADS_DEMO / "nli.npz", allow_pickle=False) as arquivo:
    pesos_demo = {chave: arquivo[chave].copy() for chave in arquivo.files}
assert pesos_demo["weight"].shape == (3, 5120) and pesos_demo["bias"].shape == (3,)
assert pesos_demo["mean"].shape == pesos_demo["scale"].shape == (5120,)
assert all(np.isfinite(v).all() for v in pesos_demo.values())
assert (pesos_demo["scale"] > 0).all() and float(pesos_demo["temperature"]) > 0
prefixo_demo = (PACOTE_DEMO / "dados/prefixo-nli.txt").read_text().rstrip("\n")
print("Cabeça aberta: W", pesos_demo["weight"].shape, "| b", pesos_demo["bias"].shape)
INDICES = list(range(30))
resultados = []
for indice in INDICES:
    entrada = pares[indice]
    print("\nENTRADA:", json.dumps(entrada, ensure_ascii=False, indent=2))
    tokens = corpo_demo.encode_nli(entrada, prefixo_demo)
    feature, _ = corpo_demo.features(tokens)  # Phi-4 produz a representação.
    saida = aplicar_cabeca(feature, pesos_demo, "nli")  # Normalização → Wz+b → logits/T → softmax → classe.
    alvo = gabaritos[indice]["expected"]  # Gabarito lido só para avaliar.
    print("SAÍDA:", saida, "| GABARITO:", alvo)
    resultados.append({"index": indice, "expected": alvo, "output": saida})
previstos = [CLASSES.index(r["output"]["classification"]) for r in resultados]
esperados = [CLASSES.index(r["expected"]) for r in resultados]
relatorio = metricas_classes(previstos, esperados)
print("MÉTRICAS:", json.dumps(relatorio, ensure_ascii=False, indent=2))
from datetime import datetime, timezone
arquivo_resultado = Path("avaliacao-" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ") + ".json")
arquivo_resultado.write_text(json.dumps({"metrics": relatorio, "predictions": resultados}, ensure_ascii=False, indent=2))
import gc
del corpo_demo
gc.collect()


## 12. Verificar o que você aprendeu

Explique, usando o notebook:

1. O que permanece congelado e quais pesos recebem gradientes?
2. Por que o alvo precisa ficar fora da entrada do modelo?
3. Por que o cache de representações pode ser reutilizado entre épocas?
4. Por que normalização, temperatura e pesos usam partições diferentes?
5. Por que reduzir a perda de treino não basta para demonstrar generalização?
6. Por que uma constante é um baseline necessário quando um evento domina a massa?
7. O que precisa acompanhar a cabeça para reproduzir sua inferência?

**Atividade:** escolha um erro da avaliação, identifique entrada, alvo e previsão e proponha um experimento que mude apenas um fator. Não altere o gabarito para concordar com o modelo e não use o teste para escolher hiperparâmetros.

### Referências do método

- [Phi-4 — relatório técnico](https://arxiv.org/abs/2412.08905): modelo base.
- [Alain e Bengio — linear probes](https://arxiv.org/abs/1610.01644): examinar representações com camadas treináveis; não valida por si só nosso classificador textual.
- [Kingma e Ba — Adam](https://arxiv.org/abs/1412.6980): atualização dos parâmetros.
- [Guo et al. — calibração](https://proceedings.mlr.press/v70/guo17a.html): calibração por temperatura.
- [Gneiting e Raftery — regras de pontuação](https://sites.stat.washington.edu/raftery/Research/PDF/Gneiting2007jasa.pdf): avaliação de previsões probabilísticas.

A implementação específica deste piloto está nos scripts versionados do GitHub. Os resultados devem ser interpretados sob as partições, contratos e condições efetivamente executados.
